# Manual Test Notebook for LLMJudgeTempCausal

Step-by-step walkthrough of the main pipeline components.

Sections:
1. Setup & Config
2. Data Loading
3. Prompt Building
4. LLM Client & Judge
5. Metrics Computation
6. Causal Analysis
7. Visualization
8. Full ExperimentRunner (quick test)

## 1. Setup & Config

In [ ]:
import logging
import sys
import os

# Ensure the project root is on the path
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.insert(0, os.path.join(project_root, "src"))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)

print("Project root:", project_root)

In [ ]:
from llmjudgetempcausal.config import (
    BackendType,
    ExperimentConfig,
    JudgeType,
    ModelConfig,
    PromptVariant,
)

# Qwen 2.5-14B on localhost:8000
model_cfg = ModelConfig(
    model_name="Qwen/Qwen2.5-14B-Instruct",
    base_url="http://localhost:8000",
    backend=BackendType.VLLM,
    model_size_label="14B",
)

# Gemma 3-12B on 10.6.32.18:8001
# vLLM requires strict user/assistant alternation; adapt_messages_for_model() handles this automatically.
gemma_model_cfg = ModelConfig(
    model_name="google/gemma-3-12b-it",
    base_url="http://10.6.32.18:8001",
    api_key="token-abc123",
    backend=BackendType.VLLM,
    model_size_label="12B",
)

config = ExperimentConfig(
    temperatures=[0.01, 0.5, 1.0, 1.5, 2.0, 3.0],
    judge_types=[JudgeType.PAIRWISE, JudgeType.SINGLE_ANSWER, JudgeType.REFERENCE_GUIDED],
    prompt_variants=[PromptVariant.BASELINE, PromptVariant.COT],
    num_repeats=15,
    sample_size=3,
    output_dir="../results_test_notebook",
    models=[model_cfg, gemma_model_cfg],
)

print("Config:", config)
print("Models:", [m.model_name for m in config.models])


## 2. Data Loading

Load MT-Bench human judgments and sample a small subset.

In [ ]:
from llmjudgetempcausal.data import load_temp_bench, sample_pairs

all_pairs = load_temp_bench(path="/home/snt/projects_lujun/LLMJudgeTempCausal/src/llmjudgetempcausal/assets/combined_dataset_with_reference_good_row_idx.json")  # 默认读 assets/mmlu_pro_judged_stream.jsonl
print(f"Total pairs loaded: {len(all_pairs)}")

pairs = sample_pairs(all_pairs, n=1, seed=42)
print(f"Sampled pairs: {len(pairs)}")

In [ ]:
# Inspect first pair
pair = pairs[0]
print(f"Question ID: {pair.question_id}")
print(f"Model A: {pair.model_a}")
print(f"Model B: {pair.model_b}")
print(f"Human winner: {pair.human_winner}")
print(f"Turn: {pair.turn}")
print(f"\nQuestion text:\n{pair.question_text[:500]}")
print(f"\nResponse A (first 300 chars):\n{pair.response_a[:500]}")
print(f"\nResponse B (first 300 chars):\n{pair.response_b[:500]}")

In [ ]:
import pandas as pd

pairs_df = pd.DataFrame([
    {
        "question_id": p.question_id,
        "model_a": p.model_a,
        "model_b": p.model_b,
        "human_winner": p.human_winner,
        "turn": p.turn,
        "question_text": p.question_text,
        "response_a": p.response_a,
        "response_b": p.response_b,
        "follow_up_question": p.follow_up_question,
    }
    for p in all_pairs
])

print(f"pairs_df shape: {pairs_df.shape}")
pairs_df.head()

## 3. Prompt Building

Build judge prompts for different judge types and prompt variants.

In [ ]:
from llmjudgetempcausal.prompts import build_messages

# Pairwise baseline — show for both models so you can compare message shapes
for cfg in [model_cfg, gemma_model_cfg]:

    msgs = build_messages(pair, JudgeType.PAIRWISE, PromptVariant.BASELINE, model_name=cfg.model_name)
    for m in msgs:
        print(f"[{m['role']}] {m['content']}")
        print()


In [ ]:
# Pairwise CoT — Gemma 3 will merge system+user into one user message
for cfg in [model_cfg, gemma_model_cfg]:

    msgs = build_messages(pair, JudgeType.PAIRWISE, PromptVariant.COT, model_name=cfg.model_name)
    print(f"  roles: {[m['role'] for m in msgs]}")
    for m in msgs:
        print(f"[{m['role']}] {m['content']}")
        print()


In [ ]:
from llmjudgetempcausal.prompts import build_messages

# Single answer baseline
for cfg in [model_cfg, gemma_model_cfg]:

    msgs = build_messages(pair, JudgeType.SINGLE_ANSWER, PromptVariant.BASELINE, model_name=cfg.model_name)
    print(f"  roles: {[m['role'] for m in msgs]}")
    for m in msgs:
        print(f"[{m['role']}] {m['content'][:300]}")
        print()


In [ ]:
from llmjudgetempcausal.prompts import build_messages

# Single answer CoT
for cfg in [model_cfg, gemma_model_cfg]:
    msgs = build_messages(pair, JudgeType.SINGLE_ANSWER, PromptVariant.COT, model_name=cfg.model_name)
    print(f"  roles: {[m['role'] for m in msgs]}")
    for m in msgs:
        print(f"[{m['role']}] {m['content']}")
        print()


In [ ]:
from llmjudgetempcausal.prompts import build_messages

# Reference-guided baseline
for cfg in [model_cfg, gemma_model_cfg]:
    msgs = build_messages(pair, JudgeType.REFERENCE_GUIDED, PromptVariant.BASELINE, model_name=cfg.model_name)
    print(f"  roles: {[m['role'] for m in msgs]}")
    for m in msgs:
        print(f"[{m['role']}] {m['content']}")
        print()


In [ ]:
from llmjudgetempcausal.prompts import build_messages

# Reference-guided CoT
for cfg in [model_cfg, gemma_model_cfg]:
    msgs = build_messages(pair, JudgeType.REFERENCE_GUIDED, PromptVariant.COT, model_name=cfg.model_name)
    print(f"  roles: {[m['role'] for m in msgs]}")
    for m in msgs:
        print(f"[{m['role']}] {m['content']}")
        print()


## 4. LLM Client & Judge

**Requires a running LLM server** (e.g. vLLM on `http://localhost:8000`).

Skip these cells if no server is available.

In [ ]:
from llmjudgetempcausal.client import LLMClient

# Qwen client (default for most tests below)
client = LLMClient(model_cfg)
print(f"Qwen client:  {client.model_name}  @ {client._get_base_url()}")

# Gemma 3 client — messages are auto-normalized inside client.generate()
gemma_client = LLMClient(gemma_model_cfg)
print(f"Gemma client: {gemma_client.model_name}  @ {gemma_client._get_base_url()}")


In [ ]:
# ── Switch between Qwen and Gemma by changing active_client ──────────────────
active_client = gemma_client          # swap to gemma_client to test Gemma 3

msgs_pairwise = build_messages(
    pair,
    JudgeType.SINGLE_ANSWER,
    PromptVariant.COT,
    which_response="b",
    model_name=active_client.model_name,   # triggers Gemma 3 normalization when needed
)

raw_output = active_client.generate(
    msgs_pairwise,
    temperature=2.0,
    top_p=0.95,
    max_tokens=1024,
    seed=46
)
print(f"[{active_client.model_name}] Raw output:")
print(raw_output)


In [ ]:
import json
import random
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from llmjudgetempcausal.prompts import build_messages
from llmjudgetempcausal.judge import parse_pairwise, parse_score, parse_judge_reason

# -- Experiment grid -----------------------------------------------------------
TEMPERATURES = [0.01, 0.5, 1.0, 1.5, 2.0, 3.0]
N_REPEATS = 10
BASE_SEED = 42
MAX_TOKENS = 1024
TOP_P = 0.95

JUDGE_COMBOS = [
    (JudgeType.PAIRWISE, PromptVariant.BASELINE),
    (JudgeType.PAIRWISE, PromptVariant.COT),
    (JudgeType.SINGLE_ANSWER, PromptVariant.BASELINE),
    (JudgeType.SINGLE_ANSWER, PromptVariant.COT),
    (JudgeType.REFERENCE_GUIDED, PromptVariant.BASELINE),
    (JudgeType.REFERENCE_GUIDED, PromptVariant.COT),
]

SEEDS = [random.Random(BASE_SEED + i).randint(0, 2**31 - 1) for i in range(N_REPEATS)]

OUTPUT_JSONL = Path(project_root) / "src" / "llmjudgetempcausal" / "assets" / f"test_main_eval_stream_{active_client.model_name}.jsonl"
OUTPUT_JSONL.parent.mkdir(parents=True, exist_ok=True)


def make_run_key(question_id: int, judge_type: str, prompt_variant: str, temperature: float, repeat_id: int) -> str:
    # One logical experiment row per key. SINGLE_ANSWER keeps A/B in the same row.
    return f"{question_id}|{judge_type}|{prompt_variant}|{temperature}|{repeat_id}"


# Resume: only count rows without row_error as completed
processed = set()
if OUTPUT_JSONL.exists():
    with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            if obj.get("row_error"):
                continue

            run_key = obj.get("run_key")
            if not run_key:
                run_key = make_run_key(
                    question_id=int(obj["question_id"]),
                    judge_type=str(obj["judge_type"]),
                    prompt_variant=str(obj["prompt_variant"]),
                    temperature=float(obj["temperature"]),
                    repeat_id=int(obj["repeat_id"]),
                )
            processed.add(run_key)

expected_total = len(TEMPERATURES) * N_REPEATS * len(JUDGE_COMBOS) * len(pairs)
print(f"Output JSONL: {OUTPUT_JSONL}")
print(f"Seeds: {SEEDS}")
print(f"Resume state: {len(processed)} / {expected_total} already completed")


with OUTPUT_JSONL.open("a", encoding="utf-8") as f:
    # Loop order requested:
    # outer: repeat_id -> middle: judge combo -> inner: temperature
    for repeat_id in tqdm(range(N_REPEATS), desc="repeat_id"):
        seed = SEEDS[repeat_id]

        for judge_type, prompt_variant in JUDGE_COMBOS:
            for temp in TEMPERATURES:
                for p in pairs:
                    run_key = make_run_key(
                        question_id=p.question_id,
                        judge_type=judge_type.value,
                        prompt_variant=prompt_variant.value,
                        temperature=temp,
                        repeat_id=repeat_id,
                    )
                    if run_key in processed:
                        continue

                    base = {
                        "run_key": run_key,
                        "question_id": p.question_id,
                        "model_a": p.model_a,
                        "model_b": p.model_b,
                        "human_winner": p.human_winner,
                        "judge_model": active_client.model_name,
                        "judge_type": judge_type.value,
                        "prompt_variant": prompt_variant.value,
                        "temperature": temp,
                        "repeat_id": repeat_id,
                        "seed": seed,
                    }

                    try:
                        if judge_type == JudgeType.SINGLE_ANSWER:
                            # SINGLE_ANSWER: score A and B, but keep them in ONE row.
                            msgs_a = build_messages(
                                p,
                                judge_type,
                                prompt_variant,
                                which_response="a",
                                model_name=active_client.model_name,
                            )
                            raw_a = active_client.generate(
                                msgs_a,
                                temperature=temp,
                                top_p=TOP_P,
                                max_tokens=MAX_TOKENS,
                                seed=seed,
                            )

                            msgs_b = build_messages(
                                p,
                                judge_type,
                                prompt_variant,
                                which_response="b",
                                model_name=active_client.model_name,
                            )
                            raw_b = active_client.generate(
                                msgs_b,
                                temperature=temp,
                                top_p=TOP_P,
                                max_tokens=MAX_TOKENS,
                                seed=seed,
                            )

                            row = {
                                **base,
                                "which_response": "both",
                                "score_a": parse_score(raw_a),
                                "score_b": parse_score(raw_b),
                                "judge_reason_a": parse_judge_reason(raw_a),
                                "judge_reason_b": parse_judge_reason(raw_b),
                                "raw_output_a": raw_a,
                                "raw_output_b": raw_b,
                                "pairwise_winner": None,
                            }

                        else:
                            msgs = build_messages(
                                p,
                                judge_type,
                                prompt_variant,
                                model_name=active_client.model_name,
                            )
                            raw = active_client.generate(
                                msgs,
                                temperature=temp,
                                top_p=TOP_P,
                                max_tokens=MAX_TOKENS,
                                seed=seed,
                            )

                            row = {
                                **base,
                                "which_response": None,
                                "score_a": None,
                                "score_b": None,
                                "judge_reason_a": None,
                                "judge_reason_b": None,
                                "raw_output_a": None,
                                "raw_output_b": None,
                                "raw_output": raw,
                                "judge_reason": parse_judge_reason(raw),
                                "pairwise_winner": parse_pairwise(raw),
                            }

                    except Exception as e:
                        row = {
                            **base,
                            "row_error": str(e),
                        }

                    f.write(json.dumps(row, ensure_ascii=False) + "\n")
                    f.flush()

                    if "row_error" not in row:
                        processed.add(run_key)


# Reload successful rows for analysis
rows = []
error_count = 0
with OUTPUT_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue

        if obj.get("row_error"):
            error_count += 1
            continue
        rows.append(obj)

exp_df = pd.DataFrame(rows)
print(f"Done. Success rows: {len(exp_df)} | Error rows: {error_count} | Completed keys: {len(processed)}/{expected_total}")
exp_df.head(10)
